# Adjoint Algorithmic Differentiation (AAD)

## Overview

Eta provides VM-native **reverse-mode** automatic differentiation using a tape
(Wengert list). When an operation sees a `TapeRef`, the VM records the forward
operation and later computes adjoints in a reverse sweep.

Eta uses standardized safety semantics for ownership, stale references,
cross-VM boundaries, non-differentiable branch policy, and domain failures.

## How Reverse-Mode AD Works

Given a scalar function `f(x₁, …, xₙ)`, reverse-mode AD computes the full
gradient `∇f` in **one** forward pass plus **one** backward pass, regardless
of the number of inputs. That makes it the right tool when *N is large and
the output is scalar* — the exact shape of a loss / objective / option price.

The recipe is mechanical:

1. **Forward pass** — execute `f` normally, but every primitive operation
   that touches a `TapeRef` records `(op, parents, primal)` onto the active
   tape. The tape is a flat array — append-only, no graph allocation, no
   closures.
2. **Seed** — set the adjoint of the output node to `1.0` (so we're computing
   `∂f/∂xᵢ`, not a vector-Jacobian product).
3. **Backward pass** — sweep the tape in reverse. For each entry, multiply
   the local partial derivative by the parent node's adjoint and accumulate
   into the children's adjoints (the chain rule).
4. **Read** — the adjoint of input node `xᵢ` is `∂f/∂xᵢ`.

| Mode | Cost of `∇f` | Best when |
|---|---|---|
| Symbolic differentiation | Expression-blow-up | Tiny, hand-tractable `f` |
| **Forward-mode** AD | `O(N)` evaluations of `f` | `N` small, many outputs |
| **Reverse-mode** AD (this) | `O(1)` evaluations of `f` | `N` large, scalar output |
| Finite differences | `O(N)` evaluations + `O(h²)` truncation error | Cross-checking only |

Memory cost of the tape is roughly **32 bytes per recorded operation** with
zero closure allocations — a 10 000-op forward pass uses ≈320 KB of tape.

## Worked Example: The Tape By Hand

Most users only ever call `grad`. To build intuition for what it actually
*does*, here is the same computation written with raw tape primitives —
exactly what `grad` does internally:

In [1]:
;; f(x, y) = x*y + sin(x)   at (x, y) = (2, 3)
;; Expected:  ∂f/∂x = y + cos(x) = 3 + cos(2) ≈ 2.5839
;;            ∂f/∂y = x          = 2

(let ((tape (tape-new)))                       ; 1. allocate tape
  (let ((x (tape-var tape 2.0))                ; 2. register inputs
        (y (tape-var tape 3.0)))
    (tape-start! tape)                         ; 3. begin recording
    (let ((out (+ (* x y) (sin x))))           ; 4. forward pass
      (tape-stop!)                             ; 5. stop recording
      (tape-backward! tape out)                ; 6. seed = 1, reverse sweep
      (println (tape-primal  tape out))        ;   ⇒ 2.909...
      (println (tape-adjoint tape x))          ;   ⇒ 2.5839...
      (println (tape-adjoint tape y)))))       ;   ⇒ 2.0

6.9093
2.58385
2


Conceptually the recorded tape looks like this (input nodes first, then one
entry per primitive, in evaluation order):

```
node 0: var      x = 2.0                       (input)
node 1: var      y = 3.0                       (input)
node 2: mul      n0 * n1     primal = 6.0      (parents: 0, 1)
node 3: sin      n0          primal = 0.909..  (parents: 0)
node 4: add      n2 + n3     primal = 6.909..  (parents: 2, 3)
```

Backward sweep, top-down through the tape (highest index first), with
`adj[4] = 1` seeded:

| Step | Node | Local rule | Adjoint propagation |
|---|---|---|---|
| 1 | `add (n4)` | `∂(a+b)/∂a = 1`, `∂/∂b = 1` | `adj[2] += 1`, `adj[3] += 1` |
| 2 | `sin (n3)` | `∂sin(x)/∂x = cos(x)` | `adj[0] += cos(2) · adj[3]` |
| 3 | `mul (n2)` | `∂(a*b)/∂a = b`, `∂/∂b = a` | `adj[0] += y · adj[2]`, `adj[1] += x · adj[2]` |

Final adjoints: `adj[x] = y + cos(x) ≈ 2.5839`, `adj[y] = x = 2.0`. ✅

> You almost never want to write tape code by hand — use `grad`. Knowing
> the mechanics, however, makes the error tags below (`:ad/stale-ref`,
> `:ad/mixed-tape`, `:ad/no-active-tape`) self-explanatory.

In [2]:
(import (only std.aad grad))

## The `grad` Helper

`grad` is the workhorse — almost every AAD program calls it.

```text
(grad f vals)  ⇒  (primal-value gradient-vector)
```

| Argument | Type | Meaning |
|---|---|---|
| `f` | `(λ x₁ … xₙ → scalar)` | Function of `n` numeric arguments. **Must return a single scalar.** |
| `vals` | `list` of length `n` | Point at which to evaluate `f` and `∇f`. |

The result is a 2-element list `(primal grad)` where `primal` is the scalar
value of `f(vals)` and `grad` is a *vector* of partial derivatives in input
order — i.e. `grad[i] = ∂f/∂xᵢ`.

### A first example, end-to-end


In [3]:

;; f(x) = x² + 3x + 1  at x = 4
;; df/dx = 2x + 3 = 11

(println (grad (lambda (x) (+ (+ (* x x) (* 3 x)) 1))
               '(4)))
;; ⇒ (29 #(11))
;;     │   └─ ∂f/∂x = 11   (1-element vector for 1-input fn)
;;     └─ f(4) = 29


(29 #(11))


### Multiple inputs

In [4]:
;; Rosenbrock f(x,y) = (1-x)² + 100·(y-x²)²
;; At (1,1) the gradient is (0, 0) — the global minimum.

(println (grad (lambda (x y)
                 (+ (* (- 1 x) (- 1 x))
                    (* 100 (* (- y (* x x))
                              (- y (* x x))))))
               '(1 1)))
;; ⇒ (0 #(0 0))

(0 #(0 0))


### What can `f` look like?

`f` can use **any** taped primitive, **any** plain control flow that doesn't
inspect a tape ref's value (see [Pitfalls](#pitfalls)), `let` / `letrec`,
recursion, higher-order combinators, and the `ad-*` / `softplus` /
`smooth-*` helpers. It cannot:

- Return a vector / list / record (output must be a single number — if you
  have a vector-valued function, see [Cookbook → Jacobians](#cookbook)).
- Read raw `if` branches that depend on `(< taped-x 0)` under the strict
  policy — use `ad-relu` / `ad-clamp` / `softplus` instead.
- Cross VM boundaries with the partially-evaluated tape ref.

> If your function naturally takes a *vector* of parameters
> `(λ θ → loss)`, wrap it: `(grad (lambda args (loss-of-list args)) θ-list)`.
> The [Cookbook](#cookbook) shows the idiomatic packing pattern.

## Non-Differentiability Policy

Real objectives often contain `abs`, `max`, `min`, `relu`, `clamp` — all of
which have **kinks** (points where the derivative doesn't exist). Eta lets
you choose how to handle them globally:

```eta
(set-aad-nondiff-policy! 'strict)        ; default: refuse to lie
(set-aad-nondiff-policy! 'zero-subgrad)  ; deterministic 0 at the kink
(aad-nondiff-policy)                     ; ⇒ 'strict   (or whatever's set)
```

Kink definitions:

| Function | Kink at |
|---|---|
| `abs(x)` | `x = 0` |
| `max(a, b)` / `min(a, b)` | `a = b` |
| `relu(x)` (via `ad-relu`) | `x = 0` |
| `clamp(x, lo, hi)` (via `ad-clamp`) | `x = lo` or `x = hi` |

Comparison semantics on taped values:

- `strict` — comparison on taped operands raises `:ad/nondiff-strict`
- `zero-subgrad` — taped operands are compared by validated primals

### Side-by-side
- Strict, raise an error.

In [5]:
;; ── strict (default) ───────────────────────────────────
(set-aad-nondiff-policy! 'strict)
(grad (lambda (x) (ad-relu x)) '(0.0))
;; ⇒ raises  :ad/nondiff-strict   (relu has no derivative at 0)

UserError: : non-differentiable point reached in strict mode

In [6]:
(grad (lambda (x) (ad-relu x)) '(1.0))
;; ⇒ (1.0 #(1.0))                 (away from the kink, fine)


(1.0 #(1.0))

In [7]:
;; ── zero-subgrad ───────────────────────────────────────
(set-aad-nondiff-policy! 'zero-subgrad)
(grad (lambda (x) (ad-relu x)) '(0.0))
;; ⇒ (0.0 #(0.0))                 (deterministic 0)

(0.0 #(0.0))

> `'zero-subgrad` is convenient but **silently lies** at kinks — your
> optimiser may get stuck there. Prefer the smooth alternatives below
> (`softplus`, `smooth-abs`, `smooth-clamp`) in production loss functions.


## Gradient Checking

Every non-trivial AD program should be cross-checked against finite
differences during development. `std.aad` ships two helpers:

```text
(check-grad        f vals [rtol] [atol] [step-scale])  ⇒  bool
(check-grad-report f vals [rtol] [atol] [step-scale])  ⇒  vector
```

`check-grad-report` returns
`#[ok max-error aad-grad fd-grad rtol atol step-scale]` — the per-component
maximum absolute error plus both gradients side by side, so a failure is
self-diagnosing.

In [8]:
(define (loss x y) (+ (* x x) (* 3 (* x y)) (* y y)))

(check-grad loss '(1.5 -2.0))

#t

In [9]:
(check-grad-report loss '(1.5 -2.0))
;; ⇒ #(#t 4.4e-9 #(-3.0 -1.0) #(-3.0 -1.0) 1e-5 1e-7 1.0)
;;       │  │      │            │
;;       │  │      │            └ central-diff gradient
;;       │  │      └ AAD gradient
;;       │  └ max |aad - fd| over all components
;;       └ pass/fail flag

#(#t 0.0 #(-3.0 0.5) #(-3.0 0.5) 1e-05 1e-07 1.0)

> If `check-grad` fails near a kink, that's expected — finite differences
> straddle the discontinuity. Either move the test point off the kink, or
> swap `ad-relu` for `softplus` and re-check.


## Primitive Coverage and Domain Rules

Taped primitives include:

- Arithmetic: `+`, `-`, `*`, `/`
- Unary math: `sin`, `cos`, `tan`, `asin`, `acos`, `atan`, `exp`, `log`, `sqrt`
- Piecewise numerics: `abs`, `min`, `max`
- Binary math: `pow`

Domain behavior for taped primitives:

- `log(x)`: `x > 0`, else `:ad/domain`
- `sqrt(x)`: `x ≥ 0`, else `:ad/domain`
- `asin(x)`, `acos(x)`: `-1 ≤ x ≤ 1`, else `:ad/domain`
- `pow(negative, non-integer exponent)`: `:ad/domain`
- `pow(0, negative)`: `:ad/domain`
- `pow(0, 0)`:
  - `strict`: `:ad/domain`
  - `zero-subgrad`: value `1`, zero subgrad
- `pow(0, positive < 1)`:
  - `strict`: `:ad/domain`
  - `zero-subgrad`: finite forward value with zero subgrad at singular base derivative

# European Option Greeks with AAD

## Overview

[`examples/european.eta`](../examples/european.eta) computes
**Black-Scholes option Greeks** — both first-order and second-order —
using Eta's built-in [tape-based reverse-mode AD](aad.md).

**Key ideas demonstrated:**

- Tape-based AD with transparent recording — plain `+`, `*`, `log`, `exp` etc.
- First-order Greeks (Delta, Vega, Theta, Rho) in a **single backward pass**
- Second-order Greeks (Gamma, Vanna, Volga) by applying `grad` to Greek expressions
- Schwarz's theorem as a built-in consistency check

```bash
etai examples/european.eta
```

> This example uses the **Black-Scholes closed-form** rather than Monte
> Carlo simulation.  The AD technique is identical — every `+`, `exp`,
> `norm-cdf` call would work the same way inside a path-level MC loop.
> The closed form lets us verify every Greek against its known analytic
> value.

---

## Market Parameters

| Parameter | Symbol | Value | Description |
|-----------|--------|-------|-------------|
| Spot price | *S* | 100.0 | Current price of the underlying |
| Strike | *K* | 90.0 | Option strike price |
| Risk-free rate | *r* | 3 % | Continuous compounding |
| Volatility | *σ* | 30 % | Annualised implied volatility |
| Maturity | *T* | 0.5 | Time to expiry in years |

These match the standard Python autograd benchmark
(`F=100, vol=0.3, K=90, T=0.5, IR=0.03`).

---

## Statistical Functions

### Normal PDF — `norm-pdf`

$$\varphi(x) = \frac{1}{\sqrt{2\pi}} \, e^{-x^2/2}$$

In [10]:
   (define branch-tape #f)
    (defun branch-primal (x)
      (if (and branch-tape (tape-ref? x))
        (tape-ref-value-of branch-tape x)
        x))
    
    ;; ── Normal PDF: φ(x) = (1/√2π) · e^{-x²/2} ─────────────────
    (define inv-sqrt-2pi 0.3989422804014327)

    (defun norm-pdf (x)
      (* inv-sqrt-2pi (exp (* -0.5 (* x x)))))

    ;; ── Normal CDF: Abramowitz & Stegun 26.2.17 ──────────────────
    ;; Maximum absolute error < 7.5 × 10⁻⁸
    ;; All arithmetic is plain +, *, / — the tape records every step.
    (defun norm-cdf (x)
      (let ((xv (branch-primal x)))
        (if (< xv 0)
          (- 1.0 (norm-cdf (* -1 x)))
          (let ((t (/ 1.0 (+ 1.0 (* 0.2316419 x)))))
            (- 1.0 (* (* inv-sqrt-2pi (exp (* -0.5 (* x x))))
                (* t (+ 0.319381530
                    (* t (+ -0.356563782
                        (* t (+ 1.781477937
                            (* t (+ -1.821255978
                                (* t 1.330274429)))))))))))))))


## Black-Scholes Formulas

### d₁ and d₂

$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2) \cdot T}{\sigma \sqrt{T}}$$

$$d_2 = d_1 - \sigma\sqrt{T}$$


In [11]:
(defun bs-d1 (S K r sigma T)
  (/ (+ (log (/ S K))
        (* (+ r (/ (* sigma sigma) 2)) T))
     (* sigma (sqrt T))))


### Call Price

$$C = S \cdot \Phi(d_1) - K \cdot e^{-rT} \cdot \Phi(d_2)$$

In [12]:
    (defun bs-d1 (S K r sigma T)
      (/ (+ (log (/ S K))
          (* (+ r (/ (* sigma sigma) 2)) T))
        (* sigma (sqrt T))))

    ;; C = S·Φ(d₁) − K·e^{-rT}·Φ(d₂)
    (defun bs-call-price (S K r sigma T)
      (let ((d1  (bs-d1 S K r sigma T))
          (svt (* sigma (sqrt T))))
        (let ((d2 (- d1 svt)))
          (- (* S (norm-cdf d1))
            (* (* K (exp (* -1 (* r T))))
              (norm-cdf d2))))))


> All functions use **plain arithmetic** — `+`, `-`, `*`, `/`, `log`,
> `exp`, `sqrt`.  When called inside `grad`, the arguments are TapeRefs
> and the VM records every operation onto the active tape.  No `d+`,
> `dlog`, `dexp` wrappers are needed.

## First-Order Greeks

A single call to `grad` on `bs-call-price` with inputs
`(S, K, r, σ, T)` produces the price **and** all five partial
derivatives in one backward pass:

In [13]:
(grad (lambda (S K r sigma T)
        (bs-call-price S K r sigma T))
      '(100.0 90.0 0.03 0.30 0.5))
;; => (price #(∂C/∂S  ∂C/∂K  ∂C/∂r  ∂C/∂σ  ∂C/∂T))

(14.8807 #(0.749668 -0.667623 30.0431 22.486 8.54839))


## Second-Order Greeks

### Grad-on-Greek

To compute Gamma (∂²C/∂S²), we express **Delta as a function**
and differentiate it:

$$\Delta(S,K,r,\sigma,T) = \Phi(d_1)$$

In [14]:
(defun bs-delta-fn (S K r sigma T)
  (norm-cdf (bs-d1 S K r sigma T)))

(grad (lambda (S K r sigma T)
        (bs-delta-fn S K r sigma T))
      '(100.0 90.0 0.03 0.30 0.5))
;; => (delta #(∂Δ/∂S  ∂Δ/∂K  ∂Δ/∂r  ∂Δ/∂σ  ∂Δ/∂T))
;;             Gamma                   Vanna

(0.74967 #(0.0149906 -0.0166563 0.749531 -0.488997 -0.101727))

Similarly, for Volga (∂²C/∂σ²), express **Vega as a function**:

$$\mathcal{V}(S,K,r,\sigma,T) = S \cdot \varphi(d_1) \cdot \sqrt{T}$$

In [15]:
(defun bs-vega-fn (S K r sigma T)
  (* S (* (norm-pdf (bs-d1 S K r sigma T)) (sqrt T))))

(grad (lambda (S K r sigma T)
        (bs-vega-fn S K r sigma T))
      '(100.0 90.0 0.03 0.30 0.5))
;; => (vega #(∂V/∂S  ∂V/∂K  ∂V/∂r  ∂V/∂σ  ∂V/∂T))
;;            Vanna                  Volga

(22.4859 #(-0.488997 0.793173 -35.6928 23.2861 27.3302))

In [16]:
(import std.jupyter)
(define params '(100.0 90.0 0.03 0.30 0.5))
(let* ((result (grad (lambda (S K r sigma T)
                       (bs-call-price S K r sigma T))
                     params))
       (price (car result))
       (g (cadr result))
       (tbl (make-fact-table 'section 'metric 'value)))
  (fact-table-insert! tbl "Raw" "Price" price)
  (fact-table-insert! tbl "Raw" "Delta" (vector-ref g 0))
  (fact-table-insert! tbl "Raw" "Rho" (vector-ref g 2))
  (fact-table-insert! tbl "Raw" "Vega" (vector-ref g 3))
  (fact-table-insert! tbl "Raw" "-Theta" (vector-ref g 4))
  (fact-table-insert! tbl "Normalised" "Vega (per 1% vol)" (/ (vector-ref g 3) 100))
  (fact-table-insert! tbl "Normalised" "Rho (per 1% rate)" (/ (vector-ref g 2) 100))
  (fact-table-insert! tbl "Normalised" "Theta (per day)" (* -1 (/ (vector-ref g 4) 365.25)))
  (jupyter:table tbl))

section,metric,value
"""Raw""","""Price""",14.8807
"""Raw""","""Delta""",0.749668
"""Raw""","""Rho""",30.0431
"""Raw""","""Vega""",22.486
"""Raw""","""-Theta""",8.54839
"""Normalised""","""Vega (per 1% vol)""",0.22486
"""Normalised""","""Rho (per 1% rate)""",0.300431
"""Normalised""","""Theta (per day)""",-0.0234042


In [20]:
(let* ((delta-result (grad (lambda (S K r sigma T)
                             (bs-delta-fn S K r sigma T))
                           params))
       (vega-result  (grad (lambda (S K r sigma T)
                             (bs-vega-fn S K r sigma T))
                           params))
       (delta (car delta-result))
       (dg    (cadr delta-result))
       (vega  (car vega-result))
       (vg    (cadr vega-result))
       (tbl   (make-fact-table 'section 'metric 'value)))
  (fact-table-insert! tbl "grad(Delta)" "Delta" delta)
  (fact-table-insert! tbl "grad(Delta)" "Gamma (dDelta/dS)" (vector-ref dg 0))
  (fact-table-insert! tbl "grad(Delta)" "Vanna (dDelta/dsigma)" (vector-ref dg 3))

  (fact-table-insert! tbl "grad(Vega)" "Vega" vega)
  (fact-table-insert! tbl "grad(Vega)" "Vanna (dVega/dS)" (vector-ref vg 0))
  (fact-table-insert! tbl "grad(Vega)" "Volga (dVega/dsigma)" (vector-ref vg 3))

  (fact-table-insert! tbl "Schwarz Check" "Vanna difference"
                      (- (vector-ref dg 3) (vector-ref vg 0)))
  (jupyter:table tbl))

section,metric,value
"""grad(Delta)""","""Delta""",0.74967
"""grad(Delta)""","""Gamma (dDelta/dS)""",0.0149906
"""grad(Delta)""","""Vanna (dDelta/dsigma)""",-0.488997
"""grad(Vega)""","""Vega""",22.4859
"""grad(Vega)""","""Vanna (dVega/dS)""",-0.488997
"""grad(Vega)""","""Volga (dVega/dsigma)""",23.2861
"""Schwarz Check""","""Vanna difference""",-2.09989e-07
